In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [5]:
# 1. Preprocesamiento
# Quitar columnas de identificadores (no aportan al modelo)
data = df.drop(columns=['nameOrig', 'nameDest'])

# Convertir la variable categórica 'type' en columnas numéricas (one-hot)
data = pd.get_dummies(data, columns=['type'], drop_first=True)

# 2. Separar características (X) y variable objetivo (y)
X = data.drop(columns=['isFraud'])
y = data['isFraud']

# 3. Dividir en entrenamiento (80%) y prueba (20%)
#    stratify=y mantiene la misma proporción de fraudes en ambos grupos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Normalizar las características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Entrenar el modelo de regresión logística
#    class_weight='balanced' es clave: el fraude es MUY poco frecuente
modelo = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo.fit(X_train_scaled, y_train)

# 6. Predecir sobre el conjunto de prueba
y_pred = modelo.predict(X_test_scaled)

# 7. Evaluación final del modelo
print("=== Evaluación final del modelo ===\n")
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación (precision, recall, f1):")
print(classification_report(y_test, y_pred, digits=4))

=== Evaluación final del modelo ===

Matriz de confusión:
[[1209017   61864]
 [     42    1601]]

Reporte de clasificación (precision, recall, f1):
              precision    recall  f1-score   support

           0     1.0000    0.9513    0.9750   1270881
           1     0.0252    0.9744    0.0492      1643

    accuracy                         0.9514   1272524
   macro avg     0.5126    0.9629    0.5121   1272524
weighted avg     0.9987    0.9514    0.9738   1272524

